In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\ramyayoub\Desktop\IAAC\Master\Semester-3\Graph ML -- DOCUMENTS\Graph ML --Ramy\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(Helper.Version())
renderer = "vscode"

The version that you are using (0.9.26) is EQUAL TO the latest version available on PyPI.


In [3]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)


In [4]:
plan_analysis = Topology.ByBREPPath(r"C:\Users\ramyayoub\Desktop\IAAC\Master\Semester-3\Graph ML -- DOCUMENTS\Graph ML --Ramy\Graph ML -- Assignment\Assignment-02\01-Assets\analysis_plan_clean.brep")

In [28]:
b_r = Wire.BoundingRectangle(plan_analysis)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange1 = list(range(0,int(width)+5,5))
vRange1 = list(range(0,int(length)+5,5))

uRange2 = list(range(0,int(width)+1,1))
vRange2 = list(range(0,int(length)+1,1))
grid1 = Grid.VerticesByDistances(plan_analysis, clip=True, uRange=uRange1, vRange=vRange1)
grid2 = Grid.EdgesByDistances(plan_analysis, clip=True, uRange=uRange2, vRange=vRange2)

In [29]:
shell = Topology.Slice(plan_analysis, grid2)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

In [30]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
analysis_graph = Graph.ByTopology(shell)

In [31]:
g_verts = Graph.Vertices(analysis_graph)
iso_verts = Topology.Vertices(grid1)

## 11. Spatial Intelligence through Isovists and Graph Analysis

## Visibility Graph Analysis
### Create Isovists
* Time consuming (about 20 minutes!)
* Will print out errors. Ignore.

In [32]:
valid_verts = [v for v in iso_verts if Vertex.IsInternal2D(v, plan_analysis)]
print(f"Total iso_verts: {len(iso_verts)}")
print(f"Valid (inside plan): {len(valid_verts)}")

Total iso_verts: 39
Valid (inside plan): 39


In [33]:
isovists = []
for v in iso_verts:
    isovist = Face.Isovist(plan_analysis, v)
    isovists.append(isovist)

Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Shell.ExternalBoundary - Error: External boundary could not be found. Returning None.
Face.Isovist - Error: Could not create isovist. Returning None.
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Face.RemoveCollinearEdges - Error: The input face parameter is not a valid face. Returning None.
caller name: Isovist
Face.RemoveCollinearEdge

In [34]:
successful = [i for i in isovists if i is not None]
print(f"Successful: {len(successful)} / {len(isovists)}")

Successful: 37 / 39


In [35]:
Topology.Show(plan_analysis, isovists,
              faceColorKey="cp_color",
              faceOpacity=0.6,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### Compute the visibility of each isovist viewpoint
* Calculate how many other points of the dense grid are within each isovist's face.

In [ ]:
new_verts = []
n_list = []
for i, iso in enumerate(isovists):
    if iso: # Skip is iso is None
        v = iso_verts[i]
        b_list = Vertex.IsInternal2D(g_verts, iso)
        b_list = [b for b in b_list if b]
        n = len(b_list)
        n_list.append(n)
        d = Dictionary.ByKeyValue("visibility", n)
        v = Topology.SetDictionary(v, d)
        new_verts.append(v)


Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error: The input face parameter is not a topologic face. Returning None.
Face.ExternalBoundary - Error:

In [44]:
# Check coordinates of first isovist vertex
print("iso_verts sample:")
for v in iso_verts[:3]:
    print(f"  ({Vertex.X(v):.2f}, {Vertex.Y(v):.2f})")

# Check coordinates of first graph vertex
print("\ng_verts sample:")
for v in g_verts[:3]:
    print(f"  ({Vertex.X(v):.2f}, {Vertex.Y(v):.2f})")

# Check first valid isovist
for iso in isovists:
    if iso is not None:
        print(f"\nFirst valid isovist type: {type(iso)}")
        verts = Topology.Vertices(iso)
        print(f"Isovist vertices sample:")
        for v in verts[:3]:
            print(f"  ({Vertex.X(v):.2f}, {Vertex.Y(v):.2f})")
        break

iso_verts sample:
  (18.35, 10.50)
  (18.35, 15.50)
  (18.35, 20.50)

g_verts sample:
  (15.85, 26.00)
  (64.85, 35.81)
  (63.85, 35.25)

First valid isovist type: <class 'topologic_core.Cluster'>
Isovist vertices sample:
  (33.72, 32.90)
  (33.28, 32.90)
  (33.28, 34.36)


### Transfer/Interpolate values from the new graph vertices to the original graph vertices

In [45]:
new_verts = []
n_list = []
for i, iso in enumerate(isovists):
    if iso is not None:
        try:
            # Extract face from cluster
            iso_faces = Topology.Faces(iso)
            if not iso_faces:
                continue
            iso_face = iso_faces[0]  # get the actual face
            
            v = iso_verts[i]
            count = 0
            for gv in g_verts:
                result = Vertex.IsInternal2D(gv, iso_face)
                if result:
                    count += 1
            n_list.append(count)
            d = Dictionary.ByKeyValue("visibility", count)
            v = Topology.SetDictionary(v, d)
            new_verts.append(v)
        except:
            pass

print(f"Valid visibility points: {len(new_verts)}")
print(f"Max visibility: {max(n_list) if n_list else 0}")
print(f"Min visibility: {min(n_list) if n_list else 0}")
print(f"Mean visibility: {round(sum(n_list)/len(n_list), 2) if n_list else 0}")

Valid visibility points: 36
Max visibility: 411
Min visibility: 18
Mean visibility: 200.39


In [46]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

### Derive the colour of each vertex based on the interpolated value

In [47]:
minValue = min(n_list)
maxValue = max(n_list)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

### Transfer the information from the graph vertices to the faces of the original shell

In [48]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [49]:
Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)